In [ ]:
import numpy as np
import math
from dataclasses import dataclass
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
def stumpf_S(z):
    """
    Stumpf function S(z).

    TODO: figure out what tolerance for |z|>tol I actually should use
    """
    if z > 1e-12:
        sqrt_z = np.sqrt(z)
        return (sqrt_z - np.sin(sqrt_z)) / (sqrt_z**3)
    elif z < -1e-12:
        sqrt_neg_z = np.sqrt(-z)
        return (np.sinh(sqrt_neg_z) - sqrt_neg_z) / (sqrt_neg_z**3)
    else:
        return 1/6

def stumpf_C(z):
    """
    Stumpf function C(z).

    TODO: figure out what tolerance for |z|>tol I actually should use
    """
    if z > 1e-12:
        sqrt_z = np.sqrt(z)
        return (1 - np.cos(sqrt_z)) / z
    elif z < -1e-12:
        sqrt_neg_z = np.sqrt(-z)
        return (np.cosh(sqrt_neg_z) - 1) / (-z)
    else:
        return 1/2
    
def stumpf_dSdz(z):
    """
    Stumpf function S(z) derivative with respect to z.
    """
    if np.abs(z) < 1e-2:
        # use power series expansion for small z
        return -1/math.factorial(5) + 2*z/math.factorial(7) - 3*z**2/math.factorial(9) + 4*z**3/math.factorial(11)
    
    S = stumpf_S(z)
    C = stumpf_C(z)
    
    return (C - 3*S) / (2*z)

def stumpf_dCdz(z):
    """
    Stumpf function C(z) derivative with respect to z.
    """
    if np.abs(z) < 1e-2:
        # use power series expansion for small z
        return -1/math.factorial(4) + 2*z/math.factorial(6) - 3*z**2/math.factorial(8) + 4*z**3/math.factorial(10)

    S = stumpf_S(z)
    C = stumpf_C(z)

    return (1 - z*S - 2*C) / (2*z)

In [ ]:
MU_EARTH = 3.986e14
R_EARTH = 6378e3

@dataclass
class GaussSolution:
    r1: np.ndarray
    r2: np.ndarray
    dt: float
    v1_short: np.ndarray
    v2_short: np.ndarray
    v1_long: np.ndarray
    v2_long: np.ndarray

def gauss_uv(r1_vec, r2_vec, dt, mu=MU_EARTH):
    # standarize inputs
    r1_vec = np.array(r1_vec, dtype=float)
    r2_vec = np.array(r2_vec, dtype=float)

    r1 = np.linalg.norm(r1_vec)
    r2 = np.linalg.norm(r2_vec)
    

    # step 0: compute nu, angle between two vectors
    #   a dot b     = |a|*|b|*cos(nu)
    #   |a cross b| = |a|*|b|*sin(nu)
    #   tan(nu) =  |a cross b| / a dot b
    nu_short = math.atan2(
        np.linalg.norm(np.cross(r1_vec, r2_vec)), 
        r1_vec.dot(r2_vec)
    )
    nu_long = 2*np.pi - nu_short

    # short way
    (f, g, gdot) = gauss_uv_fg_solver(r1, r2, dt, mu, nu_short)
    v1_short = (r2_vec - f*r1_vec) / g
    v2_short = (gdot*r2_vec - r1_vec) / g

    # long way
    (f, g, gdot) = gauss_uv_fg_solver(r1, r2, dt, mu, nu_long)
    v1_long = (r2_vec - f*r1_vec) / g
    v2_long = (gdot*r2_vec - r1_vec) / g

    # TODO: get perigee values and eliminate ones that cross through earth... or maybe do this by consumer of this function?

    solution = GaussSolution(
        r1,
        r2,
        dt,
        v1_short,
        v2_short,
        v1_long,
        v2_long,
    )

    return solution


def gauss_uv_fg_solver(
    r1: float, 
    r2: float, 
    dt: float, 
    mu: float, 
    nu: float
) -> tuple[float, float, float]:
    # direction of motion
    DM = np.sign(np.pi - nu)

    # step 1: from r1 and r2 and "direction of motion", evaluate the constant (eq. 5-15, eq. 5-37 BMW)
    A = DM * math.sqrt(r1 * r2 * (1 + math.cos(nu)))

    # step 2: pick a trial value for z
    #         z = deltaE^2 for elliptical
    #         z = deltaF^2 for hyperolic
    #         good initial guess is z = 0
    z = 0
    dz = 0

    # newton raphson parameters
    dt_tol = 1e-4
    nr_max_iter = 200
    nr_counter = 0

    # bisection search parameters
    bs_max_iter = 100
    bs_counter = 0

    while(True):
        if nr_counter >= nr_max_iter:
            print("Warning: max newton-raphson iterations reached when solving for z")
            break
        if bs_counter >= bs_max_iter:
            print("Warning: max bisection-search iterations reached when solving for z")
            break
        
        bs_counter += 1

        z_trial = z + dz

        # DEBUG
        # print(z_trial)

        # step 3: evaluate S and C for selected z (eq. 4-37 and eq. 4-38 BMW)
        S = stumpf_S(z_trial)
        C = stumpf_C(z_trial)

        # step 4: determine aux variable y (eq. 5-17 BMW)
        y = r1 + r2 - A * (1 - z_trial*S) / math.sqrt(C)

        # step 4.5: check if y is negative (only can happen with short way trajectories)
        #           keep halving step size until y is no longer negative
        if y < 0:
            dz = 0.5*dz
            continue
        
        # z_trial is accepted
        z = z_trial

        nr_counter += 1
        bs_counter = 0

        # step 5: determine x (eq. 5-18 BMW)
        x = math.sqrt(y / C)

        # step 6: check trial value of z by computing dt_trial (eq. 5-20 BMW)
        #         then compare to true dt 
        #         then newton-raphson iterate

        dt_trial = (S*x**3 + A*math.sqrt(y)) / math.sqrt(mu)
        dt_error = dt_trial - dt

        # DEBUG
        # print(f"time error: {dt_error}")

        if np.abs(dt_error) < dt_tol:
            break
        
        Sprime = stumpf_dSdz(z)
        Cprime = stumpf_dCdz(z)

        dtdz = 1/(math.sqrt(mu)) * ( x**3 * (Sprime - 3*S*Cprime/(2*C)) + A/8*(3*S*math.sqrt(y)/C + A/x) )
        
        # clip to prevent huge jumps
        dz =  np.clip(-dt_error / dtdz, -5, 5)

    # step 7: evaluate f, g, gdot (eq. 5-21, 5-22, 5-23 BMW)
    #         then compute v1 and v2 (eq. 5-24, 5-25 BMW)

    S = stumpf_S(z)
    C = stumpf_C(z)
    y = r1 + r2 - A * (1 - z*S) / math.sqrt(C)

    f = 1 - y/r1
    g = A*math.sqrt(y/mu)
    gdot = 1 - y/r2

    return (f, g, gdot)

In [ ]:
# test scenario
r1 = np.array([R_EARTH + 500e3, 0, 0])
r2 = np.array([-R_EARTH - 400e3, R_EARTH, 0])
dt = 45*60
solution = gauss_uv(r1, r2, dt)


fig = go.Figure()

# earth Sphere
theta = np.linspace(0, 2*np.pi, 50)
phi = np.linspace(0, np.pi, 50)
x_s = R_EARTH * np.outer(np.cos(theta), np.sin(phi))
y_s = R_EARTH * np.outer(np.sin(theta), np.sin(phi))
z_s = R_EARTH * np.outer(np.ones(50), np.cos(phi))
fig.add_trace(go.Surface(x=x_s, y=y_s, z=z_s, colorscale='Blues', opacity=0.6, showscale=False, name='Earth'))


# positions r1 and r2
fig.add_trace(go.Scatter3d(
    x=[r1[0]], y=[r1[1]], z=[r1[2]],
    mode="markers",
    marker=dict(
        size=6,
        color="red"
    ),
    name="r1"
))
fig.add_trace(go.Scatter3d(
    x=[r2[0]], y=[r2[1]], z=[r2[2]],
    mode="markers",
    marker=dict(
        size=6,
        color="green"
    ),
    name="r2"
))

# velocity vectors (need to scale them to be visible)
scale = 240

# function to create a vector trace
def plot_vector(r, v, color, name):
    end = r + v * scale
    return go.Scatter3d(
        x=[r[0], end[0]], y=[r[1], end[1]], z=[r[2], end[2]],
        mode="lines",
        line=dict(
            color=color, 
            width=6
        ),
        name=name
    )

fig.add_trace(plot_vector(r1, solution.v1_short, "red", "v1 short"))
fig.add_trace(plot_vector(r2, solution.v2_short, "green", "v2 short"))
fig.add_trace(plot_vector(r1, solution.v1_long, "red", "v1 long"))
fig.add_trace(plot_vector(r2, solution.v2_long, "green", "v2 long"))

# Layout settings
fig.update_layout(
    title="gauss problem",
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        aspectmode='data'  # Ensures Earth looks spherical
    ),
    autosize=False,
    width=1000,
    height=800,
    template='plotly_dark'
)

fig.show()

In [ ]:
def intercept_spacecraft(r1: np.ndarray, v1: np.ndarray, r2: np.ndarray, v2: np.ndarray, mu: float):
    # cycle through potential intercept points
    pass


